# SolarWise AI — Solar Energy Generation ML Analysis
### Capstone Project: Predictive Sizing & Yield Modeling for Residential Rooftop Solar in India

This notebook documents the machine learning pipeline used by **SolarWise AI** to predict annual solar electricity generation (`annual_generation_kwh`) based on:
1. **Solar Irradiance (`solar_radiation`)**: Daily Global Horizontal Irradiance (GHI in kWh/m²/day)
2. **System Sizing (`system_size_kw`)**: Installed photovoltaic panel nameplate capacity (kW)
3. **Rooftop Shading (`shading_factor`)**: Obstruction factor (1.0 = None/Low, 0.90 = Medium, 0.78 = High)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

## 1. Load & Inspect Dataset
The dataset contains 1,200 benchmark rooftop installations across 24 representative Indian cities.

In [2]:
df = pd.read_csv('../data/solar_generation_data.csv')
df.head()

In [3]:
df.describe()

## 2. Feature Selection & Train-Test Split (80/20)

In [4]:
features = ['solar_radiation', 'system_size_kw', 'shading_factor']
target = 'annual_generation_kwh'

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set:  {X_test.shape[0]} samples")

## 3. Model Training (Linear Regression)

In [5]:
model = LinearRegression()
model.fit(X_train, y_train)

for feat, coef in zip(features, model.coef_):
    print(f"Coefficient [{feat}]: {coef:.4f}")
print(f"Intercept: {model.intercept_:.4f}")

## 4. Model Evaluation (MAE, RMSE, R²)

In [6]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE):     {mae:.2f} kWh/year")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} kWh/year")
print(f"R² Score:                       {r2:.4f}")

## 5. Sample Inference Check

In [7]:
sample_case = pd.DataFrame([{
    'solar_radiation': 5.5,
    'system_size_kw': 3.0,
    'shading_factor': 1.0
}])
predicted_kwh = model.predict(sample_case)[0]
print(f"Predicted Annual Generation for 3kW array in 5.5 GHI zone: {predicted_kwh:.1f} kWh/year")